In [1]:
import torch
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import GPT2Tokenizer, GPT2ForSequenceClassification, GPT2Config
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np

from tabulate import tabulate
from tqdm import trange
import random
from torchmetrics.classification import Recall, Accuracy, AUROC, Precision

In [4]:
!wget 'https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip'
!unzip -o smsspamcollection.zip

--2026-04-10 07:32:35--  https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 502 Bad Gateway
2026-04-10 07:32:36 ERROR 502: Bad Gateway.

Archive:  smsspamcollection.zip
  inflating: SMSSpamCollection       
  inflating: readme                  


In [5]:
!unzip -o smsspamcollection.zip

Archive:  smsspamcollection.zip
  inflating: SMSSpamCollection       
  inflating: readme                  


In [6]:
!head -10 SMSSpamCollection

ham	Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
ham	Ok lar... Joking wif u oni...
spam	Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
ham	U dun say so early hor... U c already then say...
ham	Nah I don't think he goes to usf, he lives around here though
spam	FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £1.50 to rcv
ham	Even my brother is not like to speak with me. They treat me like aids patent.
ham	As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy your friends Callertune
spam	WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.
spam	H

In [7]:
!wget 'https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip'
!unzip -o smsspamcollection.zip

--2026-04-10 07:37:39--  https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 502 Bad Gateway
2026-04-10 07:37:40 ERROR 502: Bad Gateway.

Archive:  smsspamcollection.zip
  inflating: SMSSpamCollection       
  inflating: readme                  


In [8]:
file_path = 'SMSSpamCollection'
df = pd.DataFrame({'label':int(), 'text':str()}, index = [])
with open(file_path) as f:
    for line in f.readlines():
        split = line.split('\t')
        df = pd.concat([
                df,
                pd.DataFrame.from_dict({
                    'label': [1 if split[0] == 'spam' else 0],
                    'text': [split[1]]
                })
            ],
            ignore_index=True
        )
df.head()

,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...\n
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [9]:
text = df.text.values
labels = df.label.values

In [10]:
# Set to the GPT2Tokenizer and set lower case to True
tokenizer = GPT2Tokenizer.from_pretrained('gpt2', lower_case=True)
tokenizer.pad_token = '<|endoftext|>'
# Set the padding to 'left' or 'right'?
# Remember we want to use the last token's embedding to represent the entire sentence
tokenizer.padding_side = 'right'

/xiaopengli1/Courses/AIE1902/GPT1/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [11]:
def print_rand_sentence():
    '''Displays the tokens and respective IDs of a random text sample'''
    index = random.randint(0, len(text)-1)
    print(text[index])
    table = np.array([tokenizer.tokenize(text[index]),
                    tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text[index]))]).T
    print(tabulate(table,
                 headers = ['Tokens', 'Token IDs'],
                 tablefmt = 'fancy_grid'))

print_rand_sentence()

Me sef dey laugh you. Meanwhile how's my darling anjie!

╒════════════╤═════════════╕
│ Tokens     │   Token IDs │
╞════════════╪═════════════╡
│ Me         │        5308 │
├────────────┼─────────────┤
│ Ġse        │         384 │
├────────────┼─────────────┤
│ f          │          69 │
├────────────┼─────────────┤
│ Ġde        │         390 │
├────────────┼─────────────┤
│ y          │          88 │
├────────────┼─────────────┤
│ Ġlaugh     │        6487 │
├────────────┼─────────────┤
│ Ġyou       │         345 │
├────────────┼─────────────┤
│ .          │          13 │
├────────────┼─────────────┤
│ ĠMeanwhile │       11214 │
├────────────┼─────────────┤
│ Ġhow       │         703 │
├────────────┼─────────────┤
│ 's         │         338 │
├────────────┼─────────────┤
│ Ġmy        │         616 │
├────────────┼─────────────┤
│ Ġdarling   │       40003 │
├────────────┼─────────────┤
│ Ġan        │         281 │
├────────────┼─────────────┤
│ j          │          73 │
├────────────┼─

In [12]:
token_id = []
attention_masks = []

def preprocessing(input_text, tokenizer):
  '''
  Returns <class transformers.tokenization_utils_base.BatchEncoding> with the following fields:
    - input_ids: list of token ids
    - token_type_ids: list of token type ids
    - attention_mask: list of indices (0,1) specifying which tokens should considered by the model (return_attention_mask = True).
  '''
  # Use the tokenizer and the encode_plus methods to return the right data we'll need
  # Set max_length = 32 and return_tokens = 'pt'
  # Set other fields to the appropriate booleans needed
  return tokenizer.encode_plus(
      input_text,
      padding = 'max_length',
      truncation = True,
      max_length = 32,
      return_tensors = 'pt'
  )


for sample in text:
    encoding_dict = preprocessing(sample, tokenizer)
    token_id.append(encoding_dict['input_ids'])
    attention_masks.append(encoding_dict['attention_mask'])


# Gather all the torch_id, attention masks, and labels
token_id = torch.cat(token_id)
attention_masks = torch.cat(attention_masks)
labels = torch.tensor(labels)

In [13]:
def print_rand_sentence_encoding():
    '''Displays tokens, token IDs and attention mask of a random text sample'''
    index = random.randint(0, len(text) - 1)
    tokens = tokenizer.tokenize(tokenizer.decode(token_id[index]))
    print(tokens)
    token_ids = [i.numpy() for i in token_id[index]]
    attention = [i.numpy() for i in attention_masks[index]]
    table = np.array([tokens, token_ids, attention]).T
    print(
        tabulate(
            table,
            headers = ['Tokens', 'Token IDs', 'Attention Mask'],
            tablefmt = 'fancy_grid')
    )

print_rand_sentence_encoding()

['You', 'Ġare', 'Ġbeing', 'Ġcontacted', 'Ġby', 'Ġour', 'ĠDating', 'ĠService', 'Ġby', 'Ġsomeone', 'Ġyou', 'Ġknow', '!', 'ĠTo', 'Ġfind', 'Ġout', 'Ġwho', 'Ġit', 'Ġis', ',', 'Ġcall', 'Ġfrom', 'Ġyour', 'Ġmobile', 'Ġor', 'Ġland', 'line', 'Ġ0', '90', '64', '017', '305']
╒════════════╤═════════════╤══════════════════╕
│ Tokens     │   Token IDs │   Attention Mask │
╞════════════╪═════════════╪══════════════════╡
│ You        │        1639 │                1 │
├────────────┼─────────────┼──────────────────┤
│ Ġare       │         389 │                1 │
├────────────┼─────────────┼──────────────────┤
│ Ġbeing     │         852 │                1 │
├────────────┼─────────────┼──────────────────┤
│ Ġcontacted │       11237 │                1 │
├────────────┼─────────────┼──────────────────┤
│ Ġby        │         416 │                1 │
├────────────┼─────────────┼──────────────────┤
│ Ġour       │         674 │                1 │
├────────────┼─────────────┼──────────────────┤
│ ĠDating    │  

In [14]:
val_ratio = 0.2
# Recommended batch size: 16, 32. See: https://arxiv.org/pdf/1810.04805.pdf
batch_size = 16

# Indices of the train and validation splits stratified by labels
# Use train_test_split
train_idx, val_idx = train_test_split(range(len(labels)), test_size=val_ratio, stratify=labels)

# Train and validation sets
# Set to TensorDataset
train_set = TensorDataset(token_id[train_idx], attention_masks[train_idx], labels[train_idx])

val_set = TensorDataset(token_id[val_idx], attention_masks[val_idx], labels[val_idx])

# Prepare DataLoader
train_dataloader = DataLoader(train_set, batch_size = batch_size, shuffle = True)

validation_dataloader = DataLoader(val_set, batch_size = batch_size, shuffle = False)

### Load specific versions of the model

In [15]:
# Load the BertForSequenceClassification model
# Do not ouput the attentions and all hidden states

config = GPT2Config.from_pretrained('gpt2', output_attentions = False, output_hidden_states = False)

# Set to 'gpt2' (the smallest GPT2 which is 120 M parameters)
# Use the config above and set other labels as needed
model = GPT2ForSequenceClassification.from_pretrained('gpt2', config=config)

# Set the pad token id to the eos token id
model.config.pad_token_id = model.config.eos_token_id

# Recommended learning rates (Adam): 5e-5, 3e-5, 2e-5
# See: https://arxiv.org/pdf/1810.04805.pdf
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr = 5e-5,
    eps = 1e-08
)

/xiaopengli1/Courses/AIE1902/GPT1/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Set the model to the right device

In [34]:
# If on GPU, do as below
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [35]:
model = model.to(device)

# Recommended number of epochs: See: https://arxiv.org/pdf/1810.04805.pdf
epochs = 2

In [36]:
# Print all the layers of this GPT2 model and the number of parameters per layer
# If this is False, fine tune just the classifier layer and leave all other GPT2 parameters alone
# If this is True, fine tune everything
fine_tune = True

# Turn off gradients using the above
total_parameters = 0
for name, param in model.named_parameters():
    if (not fine_tune) and (name != 'score.weight'):
        param.requires_grad = False
    total_parameters += param.numel()
    print(name, param.numel(), param.shape, param.requires_grad)

assert(total_parameters == 124441344)

transformer.wte.weight 38597376 torch.Size([50257, 768]) True
transformer.wpe.weight 786432 torch.Size([1024, 768]) True
transformer.h.0.ln_1.weight 768 torch.Size([768]) True
transformer.h.0.ln_1.bias 768 torch.Size([768]) True
transformer.h.0.attn.c_attn.weight 1769472 torch.Size([768, 2304]) True
transformer.h.0.attn.c_attn.bias 2304 torch.Size([2304]) True
transformer.h.0.attn.c_proj.weight 589824 torch.Size([768, 768]) True
transformer.h.0.attn.c_proj.bias 768 torch.Size([768]) True
transformer.h.0.ln_2.weight 768 torch.Size([768]) True
transformer.h.0.ln_2.bias 768 torch.Size([768]) True
transformer.h.0.mlp.c_fc.weight 2359296 torch.Size([768, 3072]) True
transformer.h.0.mlp.c_fc.bias 3072 torch.Size([3072]) True
transformer.h.0.mlp.c_proj.weight 2359296 torch.Size([3072, 768]) True
transformer.h.0.mlp.c_proj.bias 768 torch.Size([768]) True
transformer.h.1.ln_1.weight 768 torch.Size([768]) True
transformer.h.1.ln_1.bias 768 torch.Size([768]) True
transformer.h.1.attn.c_attn.weigh

### Train the model

In [37]:
# Use torchmetrics to set up accuracy, recall, precision, and auroc
accuracy = Accuracy(task= 'binary')
recall = Recall(task= 'binary')
precision = Precision(task= 'binary')
auroc = AUROC(task= 'binary')

In [38]:
# Main training / validation loop
for _ in trange(epochs, desc = 'Epoch'):

    # ========== Training ==========

    # Set model to training mode
    model.train()

    # Tracking variables
    tr_loss = 0
    nb_tr_examples, nb_tr_steps = 0, 0

    for step, batch in enumerate(train_dataloader):
        # Put each element of batch onto the device
        batch = [x.to(device) for x in batch]

        # Unpack the batch
        token_ids, attention_masks, labels = batch

        # Set gradients to zero
        optimizer.zero_grad()

        # Forward pass
        train_output = model(input_ids=token_ids, attention_mask=attention_masks, labels=labels)

        # Backward pass
        train_output.loss.backward()
        optimizer.step()

        # Update tracking variables
        tr_loss += train_output.loss.item()
        nb_tr_examples += token_ids.size(0)
        nb_tr_steps += 1

    # ========== Validation ==========

    # Set model to evaluation mode
    model.eval()

    # Tracking variables
    val_accuracy = []
    val_precision = []
    val_recall = []
    val_auroc = []

    for batch in validation_dataloader:
        batch = [x.to(device) for x in batch]

        # Unpack the batch
        token_ids, attention_masks, labels = batch

        with torch.no_grad():
          # Forward pass
            eval_output = model(input_ids=token_ids, attention_mask=attention_masks, labels=labels)

        # Calculate validation metrics
        labels = labels.cpu()
        predicted_labels = torch.argmax(eval_output.logits, dim=1).cpu()

        val_accuracy.append(accuracy(predicted_labels, labels))
        val_recall.append(recall(predicted_labels, labels))
        val_precision.append(precision(predicted_labels, labels))
        val_auroc.append(auroc(predicted_labels, labels))

    print('\n\t - Train loss: {:.4f}'.format(tr_loss / nb_tr_steps))
    print('\t - Validation Accuracy: {:.4f}'.format(sum(val_accuracy)/len(val_accuracy)))
    print('\t - Validation Precision: {:.4f}'.format(sum(val_precision)/len(val_precision)))
    print('\t - Validation Recall: {:.4f}'.format(sum(val_recall)/len(val_recall)))
    print('\t - Validation AUROC: {:.4f}\n'.format(sum(val_auroc)/len(val_auroc)))

Epoch:   0%|          | 0/2 [00:00<?, ?it/s]/xiaopengli1/Courses/AIE1902/GPT1/lib/python3.10/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)
Epoch:  50%|█████     | 1/2 [00:06<00:06,  6.02s/it]


	 - Train loss: 0.0048
	 - Validation Accuracy: 0.9884
	 - Validation Precision: 0.8990
	 - Validation Recall: 0.8817
	 - Validation AUROC: 0.9029



Epoch: 100%|██████████| 2/2 [00:12<00:00,  6.03s/it]


	 - Train loss: 0.0021
	 - Validation Accuracy: 0.9884
	 - Validation Precision: 0.9026
	 - Validation Recall: 0.8817
	 - Validation AUROC: 0.9027



### Test on a specific sentence, see the outcome

In [ ]:
new_sentence = 'WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.'
# new_sentence = "You have been selected for an exclusive investment opportunity."
# new_sentence = "The weather looks great this weekend. Want to go hiking?"
# new_sentence = "Your appointment with Dr. Smith is confirmed for April 15th at 10:00 AM."


# We need Token IDs and Attention Mask for inference on the new sentence
test_ids = []
test_attention_mask = []

# Apply the tokenizer
encoding = preprocessing(new_sentence, tokenizer)

# Extract IDs and Attention Mask
test_ids.append(encoding['input_ids'])
test_attention_mask.append(encoding['attention_mask'])
test_ids = torch.cat(test_ids, dim = 0)
test_attention_mask = torch.cat(test_attention_mask, dim = 0)

# Forward pass, calculate logit predictions
with torch.no_grad():
    output = model(test_ids.to(device), token_type_ids = None, attention_mask = test_attention_mask.to(device))

prediction = 'Spam' if np.argmax(output.logits.cpu().numpy()).flatten().item() == 1 else 'Ham'

print('Input Sentence: ', new_sentence)
print('Predicted Class: ', prediction)

Input Sentence:  Your appointment with Dr. Smith is confirmed for April 15th at 10:00 AM.
Predicted Class:  Ham


### Questions

Question 1: Run the above by fine tuning GPT2 and the classfier head and by not doing this (using GPT2 as a feature encoder). What is the gap between this? What are the metrics we get in each case?

Solution: after the same number of epochs (2 epochs),  
  - using GPT2 as a feature encoder.
    - Train loss: 0.0309
    - Validation Accuracy: 0.9871
    - Validation Precision: 0.8590
    - Validation Recall: 0.8710
    - Validation AUROC: 0.8810
  - not using GPT2 as a feature encoder.
    - Train loss: 0.2852
    - Validation Accuracy: 0.9250
    - Validation Precision: 0.5452
    - Validation Recall: 0.4540
    - Validation AUROC: 0.6643

There is an obvious gap between using and not using GPT2 as a feature encoder. The 4 metrics of using GPT2 as a feature encoder are all much higher than those of not using GPT2.
- Train loss gap (using - not using): -0.2543
- Validation Accuracy gap (using - not using): +0.0621
- Validation Precision gap (using - not using): +0.3138
- Validation Recall gap (using - not using): +0.4170
- Validation AUROC gap (using - not using): +0.2167

In [229]:
# - Train loss: 0.0309
# 	 - Validation Accuracy: 0.9871
# 	 - Validation Precision: 0.8590
# 	 - Validation Recall: 0.8710
# 	 - Validation AUROC: 0.8810

# - Train loss: 0.2852
# 	 - Validation Accuracy: 0.9090
# 	 - Validation Precision: 0.5452
# 	 - Validation Recall: 0.4540
# 	 - Validation AUROC: 0.6643
